In [1]:
from pyspark.sql.functions import col, udf
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegressionModel
from pyspark.sql.types import DoubleType


FEATURE_COLS = [
    "amt", "log_amt", "city_pop", "customer_age", "distance_customer_merchant",
    "trans_hour", "trans_dayofweek", "trans_month", "category_index"
]

StatementMeta(, 9d7fae6f-89fa-403b-b803-f0bb320f5985, 3, Finished, Available, Finished)

### Load test features

In [2]:
df_test = spark.sql(
    "SELECT * FROM fraud_detection_lakehouse.dbo.ml_test_features"
)

# Select needed columns
df_test = df_test.select(
    "trans_num",
    "trans_date_trans_time",
    "cc_num",
    *[col(c).cast("double").alias(c) for c in FEATURE_COLS],
    col("is_fraud").cast("double")
).dropna()

StatementMeta(, 9d7fae6f-89fa-403b-b803-f0bb320f5985, 4, Finished, Available, Finished)

### Create vector

In [3]:
assembler = VectorAssembler(inputCols=FEATURE_COLS, outputCol="features")

test_vec = assembler.transform(df_test)

StatementMeta(, 9d7fae6f-89fa-403b-b803-f0bb320f5985, 5, Finished, Available, Finished)

### Load model

In [4]:
model_path = "Files/models/fraud_lr_model"
lr_model = LogisticRegressionModel.load(model_path)

StatementMeta(, 9d7fae6f-89fa-403b-b803-f0bb320f5985, 6, Finished, Available, Finished)

### Make predictions

In [5]:
predictions = lr_model.transform(test_vec)

StatementMeta(, 9d7fae6f-89fa-403b-b803-f0bb320f5985, 7, Finished, Available, Finished)

### Get fraud probability

In [6]:
get_fraud_prob = udf(lambda v: float(v[1]), DoubleType())

predictions = predictions.withColumn(
    "fraud_probability",
    get_fraud_prob(col("probability"))
)

THRESHOLD = 0.25

predictions = predictions.withColumn(
    "is_suspicious",
    (col("fraud_probability") >= THRESHOLD).cast("int")
)

StatementMeta(, 9d7fae6f-89fa-403b-b803-f0bb320f5985, 8, Finished, Available, Finished)

### Select only the needed data for analysis and reports

In [10]:
df_scored = predictions.select(
    "trans_num",
    "trans_date_trans_time",
    "cc_num",
    *FEATURE_COLS,
    "fraud_probability",
    "is_suspicious",
    "is_fraud"
)

StatementMeta(, 9d7fae6f-89fa-403b-b803-f0bb320f5985, 12, Finished, Available, Finished)

### Save the scores

In [12]:
df_scored.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fraud_predictions")

StatementMeta(, 9d7fae6f-89fa-403b-b803-f0bb320f5985, 14, Finished, Available, Finished)